# Colab 批量幻灯片预处理

先将本地生成的 `data/ingest/colab_batch.zip` 上传到 Drive 的 `SRAG` 目录。批次包包含当前项目源码，因此不需要从 GitHub 克隆。本 Notebook 逐份解析、切块并保存预处理包；不调用 Embedding API，也不写 Qdrant。

In [1]:
import os
import platform
import shutil
import subprocess
import sys

import torch
in_colab = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

print("运行环境:", "Google Colab" if in_colab else "本地或其他 Jupyter")
print("Python:", sys.version.split()[0])
print("系统:", platform.platform())
print("PyTorch:", torch.__version__)

print("CUDA 编译版本:", torch.version.cuda)
print("CUDA 可用:", cuda_available)
print("Apple MPS 可用:", mps_available)

if cuda_available:


    print("GPU:", torch.cuda.get_device_name(0))
elif in_colab:
    
    print("\n当前 Colab 会话没有分配 GPU。请新建或切换到 GPU 运行时，然后重新连接内核。")
else:
    print("\n当前单元格没有在 Colab GPU 运行时执行。请在 VS Code 的内核选择器中连接 Colab GPU。")

运行环境: Google Colab
Python: 3.13.15
系统: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu128
CUDA 编译版本: 12.8
CUDA 可用: True
Apple MPS 可用: False
GPU: Tesla T4


## 1. 挂载 Drive 并验证批次包

In [3]:
import hashlib
import json
from pathlib import Path
from zipfile import ZipFile
from google.colab import drive

drive.mount("/content/drive")
BATCH_ZIP = Path("/content/drive/MyDrive/Colab Notebooks/SRAG/colab_batch.zip")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/Colab Notebooks/SRAG/processed")
ENABLE_PDF_OCR_FALLBACK = True
if not BATCH_ZIP.is_file():
    raise FileNotFoundError(f"请先上传批次包：{BATCH_ZIP}")
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_ROOT = Path("/content/srag_batch")
INPUT_ROOT = WORK_ROOT / "inputs"
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
with ZipFile(BATCH_ZIP) as archive:
    for member in archive.infolist():
        if not (WORK_ROOT / member.filename).resolve().is_relative_to(WORK_ROOT.resolve()):
            raise ValueError(f"压缩包路径越界：{member.filename}")
    archive.extractall(WORK_ROOT)
batch = json.loads((WORK_ROOT / "batch.json").read_text(encoding="utf-8"))
files = batch.get("files", [])
if not 1 <= len(files) <= 20:
    raise ValueError("每批必须包含 1–20 个文件")
for entry in files:
    source = (INPUT_ROOT / entry["relative_path"]).resolve()
    if not source.is_relative_to(INPUT_ROOT.resolve()) or not source.is_file():
        raise ValueError(f"文件不存在或路径越界：{entry['relative_path']}")
    if source.suffix.lower() not in {".pdf", ".pptx"}:
        raise ValueError(f"不支持的格式：{source}")
    if hashlib.sha256(source.read_bytes()).hexdigest() != entry["sha256"]:
        raise ValueError(f"源文件哈希不符：{entry['relative_path']}")
    print(entry["course_id"], entry["relative_path"], entry["pages_or_slides"], "页/张")
print("结果将保存到：", DRIVE_OUTPUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CS5495 CS5495 X AI/Lecture2/Lecture2a.pdf 20 页/张
CS5495 CS5495 X AI/Lecture2/Lecture2b.pdf 21 页/张
CS5495 CS5495 X AI/Lecture3/Lecture3a.pdf 19 页/张
CS5495 CS5495 X AI/Lecture3/Lecture3b.pdf 31 页/张
结果将保存到： /content/drive/MyDrive/Colab Notebooks/SRAG/processed


## 2. 安装批次包内的项目与文档解析依赖

In [4]:
REPO_DIR = WORK_ROOT / "project"
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError("批次包缺少项目源码")

def run_checked(command: list[str]) -> None:
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"命令执行失败（退出码 {result.returncode}）：{' '.join(command)}")

print("正在安装批次包内的 SRAG 与文档解析依赖……")
run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[documents]"])
if any(Path(x["relative_path"]).suffix.lower() == ".pptx" for x in files) and not shutil.which("libreoffice"):
    run_checked(["apt-get", "update", "-qq"])
    run_checked(["apt-get", "install", "-y", "-qq", "libreoffice-impress"])
os.environ["DOCLING_DEVICE"] = "cuda" if cuda_available else "cpu"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("项目目录：", REPO_DIR)
print("Docling 设备：", os.environ["DOCLING_DEVICE"])


正在安装批次包内的 SRAG 与文档解析依赖……
项目目录： /content/srag_batch/project
Docling 设备： cuda


## 3. 逐份解析、切块和保存

每份文档单独写入 Drive；中断后重新运行可跳过已完成的有效 ZIP。

In [5]:
import time
from app.rag.slide_chunker import SlideChunker
from app.rag.slide_preprocessor import SlideDeckPreprocessor
from zipfile import BadZipFile

preprocessor = SlideDeckPreprocessor(
    output_root=Path("/content/srag_outputs"),
    chunker=SlideChunker(max_tokens=650, overlap_tokens=50),
    enable_pdf_ocr_fallback=ENABLE_PDF_OCR_FALLBACK,
    pdf_ocr_min_text_tokens=8,
)
report = []
for entry in files:
    source = (INPUT_ROOT / entry["relative_path"]).resolve()
    document_id = f"document_{entry['sha256'][:24]}"
    saved_zip = DRIVE_OUTPUT / f"{document_id}.slidedeck.zip"
    if saved_zip.is_file():
        try:
            with ZipFile(saved_zip) as existing:
                valid = existing.testzip() is None and {"deck.json", "chunks.json"} <= set(existing.namelist())
        except BadZipFile:
            valid = False
        if valid:
            item = {"source": entry["relative_path"], "document_id": document_id, "status": "existing", "bundle": saved_zip.name}
            report.append(item)
            (DRIVE_OUTPUT / "batch_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
            print("已有结果：", item["source"])
            continue
    started = time.perf_counter()
    try:
        result = preprocessor.preprocess(source)
        if result.deck.document_id != document_id:
            raise RuntimeError("预处理文档 ID 与源文件哈希不一致")
        local_zip = Path(shutil.make_archive(f"/content/{document_id}.slidedeck", "zip", result.bundle_dir))
        partial_zip = saved_zip.with_suffix(saved_zip.suffix + ".tmp")
        shutil.copy2(local_zip, partial_zip)
        partial_zip.replace(saved_zip)
        item = {
            "source": entry["relative_path"], "document_id": document_id,
            "status": "ok", "bundle": saved_zip.name,
            "slides": len(result.deck.slides), "chunks": len(result.chunking.chunks),
            "screenshots": sum(bool(s.page_image_path) for s in result.deck.slides),
            "empty_text_chunks": sum(not c.display_text.strip() for c in result.chunking.chunks),
            "seconds": round(time.perf_counter() - started, 1),
        }
        report.append(item)
        print("完成：", item)
    except Exception as exc:
        item = {"source": entry["relative_path"], "document_id": document_id, "status": "failed", "error": str(exc)}
        report.append(item)
        print("失败：", item)
    (DRIVE_OUTPUT / "batch_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("报告：", DRIVE_OUTPUT / "batch_report.json")

完成： {'source': 'CS5495 X AI/Lecture2/Lecture2a.pdf', 'document_id': 'document_5f9ca97dd3e5d44a3e201cdb', 'status': 'ok', 'bundle': 'document_5f9ca97dd3e5d44a3e201cdb.slidedeck.zip', 'slides': 20, 'chunks': 31, 'screenshots': 20, 'empty_text_chunks': 0, 'seconds': 20.0}
完成： {'source': 'CS5495 X AI/Lecture2/Lecture2b.pdf', 'document_id': 'document_bf28bc611cc82129a0de8d08', 'status': 'ok', 'bundle': 'document_bf28bc611cc82129a0de8d08.slidedeck.zip', 'slides': 21, 'chunks': 37, 'screenshots': 21, 'empty_text_chunks': 0, 'seconds': 10.6}
完成： {'source': 'CS5495 X AI/Lecture3/Lecture3a.pdf', 'document_id': 'document_7dea2f628cb9374f5f98774a', 'status': 'ok', 'bundle': 'document_7dea2f628cb9374f5f98774a.slidedeck.zip', 'slides': 19, 'chunks': 23, 'screenshots': 19, 'empty_text_chunks': 0, 'seconds': 12.5}
完成： {'source': 'CS5495 X AI/Lecture3/Lecture3b.pdf', 'document_id': 'document_b4d00e523d059bdfa5ccbc43', 'status': 'ok', 'bundle': 'document_b4d00e523d059bdfa5ccbc43.slidedeck.zip', 'slides'

## 4. 回传本地

从 Drive 的 `SRAG/processed` 下载 `.slidedeck.zip` 与 `batch_report.json`。解压后核对页数、截图和 Chunk，再在本地调用 `app.rag.slide_embedder` 生成向量包并导入 Qdrant。